# 08 — Auditoría y reconciliación del pipeline de abastecimiento Tostao

**Propósito:** validar la integridad de los artefactos del Notebook 07 y documentar la diferencia entre los resultados Q90 reportados en las iteraciones anteriores.

Este notebook no vuelve a entrenar ni modifica el modelo. Comprueba consistencia de datos, llaves, pronósticos, escenarios, política de inventario y resúmenes. Si faltan insumos históricos para reproducir una versión anterior, lo registra como limitación y no inventa una reconciliación.

**Regla de uso:** ejecutar desde la raíz del proyecto. Los archivos de `outputs/` son el snapshot sometido a auditoría.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Ajusta estas rutas si ejecutas el notebook fuera de la raíz del proyecto.
ROOT = Path.cwd()
if not (ROOT / "outputs").exists() and (ROOT / "tostao_supply_chain_project" / "outputs").exists():
    ROOT = ROOT / "tostao_supply_chain_project"
OUTPUT_DIR = ROOT / "outputs"
DATA_DIR = ROOT / "data"
# Fallback útil cuando el proyecto y los CSV originales están en carpetas hermanas.
if not (DATA_DIR / "inventario_actual.csv").exists() and (ROOT.parent / "inventario_actual.csv").exists():
    DATA_DIR = ROOT.parent

KEYS = ["id_tienda", "id_producto"]
REQUIRED_OUTPUTS = [
    "forecast_ets_diario_7d.csv", "forecast_ets_semanal_sku_tienda.csv",
    "errores_semanales_backtesting_ets.csv", "distribucion_demanda_escenarios.csv",
    "recomendacion_abastecimiento_Q90.csv", "comparacion_politicas_Q85_Q90_Q95.csv",
    "resumen_politicas_abastecimiento.csv", "resumen_ejecucion.csv"
]
missing = [f for f in REQUIRED_OUTPUTS if not (OUTPUT_DIR / f).exists()]
print("Raíz:", ROOT)
print("Datos:", DATA_DIR)
print("Salidas faltantes:", missing)
if missing:
    raise FileNotFoundError(f"Faltan salidas del pipeline: {missing}")
frames = {Path(f).stem: pd.read_csv(OUTPUT_DIR / f) for f in REQUIRED_OUTPUTS}
for name, frame in frames.items():
    print(f"{name}: {frame.shape}")

Raíz: /mnt/data/tostao_supply_chain_project
Datos: /mnt/data
Salidas faltantes: []
forecast_ets_diario_7d: (1120, 5)
forecast_ets_semanal_sku_tienda: (160, 3)
errores_semanales_backtesting_ets: (480, 8)
distribucion_demanda_escenarios: (160, 9)
recomendacion_abastecimiento_Q90: (160, 11)
comparacion_politicas_Q85_Q90_Q95: (480, 11)
resumen_politicas_abastecimiento: (3, 5)
resumen_ejecucion: (9, 2)


## 1. Integridad estructural y llaves

Se verifica que las tablas estén al nivel esperado y no tengan llaves duplicadas. La unidad de decisión es una combinación única **tienda–producto**.

In [2]:
daily = frames["forecast_ets_diario_7d"]
weekly = frames["forecast_ets_semanal_sku_tienda"]
errors = frames["errores_semanales_backtesting_ets"]
distribution = frames["distribucion_demanda_escenarios"]
recommendations = frames["recomendacion_abastecimiento_Q90"]
comparison = frames["comparacion_politicas_Q85_Q90_Q95"]
summary_saved = frames["resumen_politicas_abastecimiento"]

checks = []
def add_check(name, passed, detail, severity="ERROR"):
    checks.append({"control": name, "estado": "OK" if bool(passed) else severity, "detalle": str(detail)})

for name, df in [("forecast semanal", weekly), ("distribución", distribution), ("recomendación Q90", recommendations)]:
    add_check(f"Llave única — {name}", not df.duplicated(KEYS).any(), f"filas={len(df)}, duplicados={int(df.duplicated(KEYS).sum())}")
add_check("Comparación tiene 3 políticas por SKU–tienda", len(comparison) == len(weekly)*3 and comparison.groupby(KEYS)["politica"].nunique().eq(3).all(), f"filas={len(comparison)}, combinaciones={comparison[KEYS].drop_duplicates().shape[0]}")
add_check("Diario tiene 7 fechas por SKU–tienda", daily.groupby(KEYS).size().eq(7).all(), f"filas={len(daily)}, combinaciones={daily[KEYS].drop_duplicates().shape[0]}")
add_check("No hay pronósticos negativos", (daily["demanda_pronosticada"] >= 0).all() and (weekly["demanda_pronosticada_7d"] >= 0).all(), "Se revisaron pronósticos diarios y semanales")
add_check("Residuos finitos", np.isfinite(errors["error_relativo"]).all(), f"residuos={len(errors)}")

checks_df = pd.DataFrame(checks)
display(checks_df)

,control,estado,detalle
0,Llave única — forecast semanal,OK,"filas=160, duplicados=0"
1,Llave única — distribución,OK,"filas=160, duplicados=0"
2,Llave única — recomendación Q90,OK,"filas=160, duplicados=0"
3,Comparación tiene 3 políticas por SKU–tienda,OK,"filas=480, combinaciones=160"
4,Diario tiene 7 fechas por SKU–tienda,OK,"filas=1120, combinaciones=160"
5,No hay pronósticos negativos,OK,Se revisaron pronósticos diarios y semanales
6,Residuos finitos,OK,residuos=480


## 2. Coherencia del pronóstico diario y semanal

El pronóstico semanal debe ser exactamente la suma de los siete pronósticos diarios para cada combinación SKU–tienda.

In [3]:
daily_sum = daily.groupby(KEYS, as_index=False)["demanda_pronosticada"].sum().rename(columns={"demanda_pronosticada":"suma_diaria_7d"})
forecast_check = weekly.merge(daily_sum, on=KEYS, how="outer", validate="one_to_one", indicator=True)
forecast_check["diferencia"] = forecast_check["demanda_pronosticada_7d"] - forecast_check["suma_diaria_7d"]
max_forecast_diff = forecast_check["diferencia"].abs().max()
add_check("Suma diaria = pronóstico semanal", forecast_check["_merge"].eq("both").all() and max_forecast_diff < 1e-8, f"diferencia absoluta máxima={max_forecast_diff:.12g}")
display(forecast_check.head())

,id_tienda,id_producto,demanda_pronosticada_7d,suma_diaria_7d,_merge,diferencia
0,STORE_01,PROD_001,49.270468,49.270468,both,0.000000e+00
1,STORE_01,PROD_002,65.009761,65.009761,both,-1.421085e-14
2,STORE_01,PROD_003,61.682972,61.682972,both,7.105427e-15
3,STORE_01,PROD_004,49.347698,49.347698,both,0.000000e+00
4,STORE_01,PROD_005,72.368104,72.368104,both,0.000000e+00


## 3. Conciliación con inventario y cálculo del pedido

La recomendación debe cubrir exactamente las 160 combinaciones del universo de datos. El pedido es `max(inventario_objetivo - stock_actual, 0)`. Se valida además que el stock publicado coincida con el archivo de inventario.

In [4]:
inventory_path = DATA_DIR / "inventario_actual.csv"
if inventory_path.exists():
    inventory = pd.read_csv(inventory_path)
    for c in KEYS: inventory[c] = inventory[c].astype(str).str.strip()
    for df in [weekly, recommendations, comparison]:
        for c in KEYS: df[c] = df[c].astype(str).str.strip()
    inv_keys = set(map(tuple, inventory[KEYS].drop_duplicates().to_numpy()))
    weekly_keys = set(map(tuple, weekly[KEYS].drop_duplicates().to_numpy()))
    add_check("Universo forecast = inventario", inv_keys == weekly_keys, f"inventario={len(inv_keys)}, forecast={len(weekly_keys)}, solo inventario={len(inv_keys-weekly_keys)}, solo forecast={len(weekly_keys-inv_keys)}")
    rec_inv = recommendations.merge(inventory[KEYS+['stock_actual']], on=KEYS, how='left', suffixes=('', '_fuente'), validate='one_to_one')
    stock_match = rec_inv['stock_actual_fuente'].notna().all() and np.allclose(rec_inv['stock_actual'], rec_inv['stock_actual_fuente'])
    add_check("Stock recomendado = inventario fuente", stock_match, f"filas sin match={int(rec_inv['stock_actual_fuente'].isna().sum())}")
else:
    add_check("Inventario fuente disponible", False, f"No se encontró {inventory_path}", "NO EVALUADO")

order_math = np.maximum(recommendations['inventario_objetivo'] - recommendations['stock_actual'], 0)
add_check("Pedido = objetivo menos stock", np.array_equal(recommendations['pedido_sugerido'].to_numpy(), order_math.astype(int).to_numpy()), f"pedido total={int(recommendations['pedido_sugerido'].sum())}")
add_check("Pedido no negativo", (recommendations['pedido_sugerido'] >= 0).all(), f"mínimo={recommendations['pedido_sugerido'].min()}")
add_check("Todas las filas Q90", recommendations['politica'].eq('Q90').all(), f"políticas={recommendations['politica'].unique().tolist()}")
add_check("Recomendación Q90 coincide con tabla comparativa", len(recommendations)==len(comparison[comparison['politica']=='Q90']) and recommendations.sort_values(KEYS)['pedido_sugerido'].reset_index(drop=True).equals(comparison[comparison['politica']=='Q90'].sort_values(KEYS)['pedido_sugerido'].reset_index(drop=True)), "Comparación de cantidades por combinación")


## 4. Recalcular el resumen Q85/Q90/Q95

Se vuelve a agregar la tabla detallada de políticas y se compara con el resumen guardado. Esto detecta resúmenes desactualizados o generados con otra corrida.

In [5]:
summary_recalc = (comparison.groupby('politica', as_index=False)
    .agg(pedido_total_unidades=('pedido_sugerido','sum'),
         inventario_objetivo_total=('inventario_objetivo','sum'),
         combinaciones_con_pedido=('pedido_sugerido', lambda s: int((s>0).sum())),
         costo_esperado_estimado=('costo_esperado_estimado','sum')))
summary_compare = summary_saved.merge(summary_recalc, on='politica', suffixes=('_guardado','_recalculado'), validate='one_to_one')
for col in ['pedido_total_unidades','inventario_objetivo_total','combinaciones_con_pedido','costo_esperado_estimado']:
    summary_compare[f'diff_{col}'] = summary_compare[f'{col}_guardado'] - summary_compare[f'{col}_recalculado']
summary_ok = True
for col in ['pedido_total_unidades','inventario_objetivo_total','combinaciones_con_pedido']:
    summary_ok &= summary_compare[f'diff_{col}'].eq(0).all()
summary_ok &= np.allclose(summary_compare['diff_costo_esperado_estimado'], 0, atol=1e-6)
add_check("Resumen guardado = resumen recalculado", summary_ok, "Diferencias en columnas diff_* deben ser cero")
display(summary_compare)
print("Resumen recalculado:")
display(summary_recalc)

,politica,pedido_total_unidades_guardado,inventario_objetivo_total_guardado,combinaciones_con_pedido_guardado,costo_esperado_estimado_guardado,pedido_total_unidades_recalculado,inventario_objetivo_total_recalculado,combinaciones_con_pedido_recalculado,costo_esperado_estimado_recalculado,diff_pedido_total_unidades,diff_inventario_objetivo_total,diff_combinaciones_con_pedido,diff_costo_esperado_estimado
0,Q85,13870,17184,156,498152.669722,13870,17184,156,498152.669722,0,0,0,0.000000e+00
1,Q90,13972,17286,156,475635.582857,13972,17286,156,475635.582857,0,0,0,5.820766e-11
2,Q95,13981,17295,156,474363.033779,13981,17295,156,474363.033779,0,0,0,0.000000e+00


Resumen recalculado:


,politica,pedido_total_unidades,inventario_objetivo_total,combinaciones_con_pedido,costo_esperado_estimado
0,Q85,13870,17184,156,498152.669722
1,Q90,13972,17286,156,475635.582857
2,Q95,13981,17295,156,474363.033779


## 5. Auditoría de residuos y escenarios

El diseño esperado en el snapshot es de 480 errores semanales (160 SKU–tienda × 3 folds), con 60 observaciones por producto (20 tiendas × 3 folds). Se comprueba la cobertura, sin asumir que los errores representan una muestra grande o independiente.

In [6]:
fold_counts = errors.groupby('fold').size().to_dict()
product_counts = errors.groupby('id_producto').size()
add_check("480 residuos semanales", len(errors)==480, f"total={len(errors)}, por fold={fold_counts}", "ADVERTENCIA")
add_check("60 residuos por producto", product_counts.eq(60).all(), f"conteos={product_counts.to_dict()}", "ADVERTENCIA")
add_check("Distribución cubre 160 SKU–tienda", len(distribution)==len(weekly) and not distribution.duplicated(KEYS).any(), f"filas distribución={len(distribution)}")
add_check("Pool de residuos documentado", distribution['fuente_residuos'].isin(['producto','global']).all() and distribution['n_residuos'].notna().all(), f"fuentes={distribution['fuente_residuos'].value_counts().to_dict()}")
add_check("Forecast sin fallback", daily['modelo'].eq('ETS_Aditivo_7').all(), f"modelos={daily['modelo'].value_counts().to_dict()}", "ADVERTENCIA")


## 6. Reconciliación histórica del total Q90

En iteraciones anteriores quedaron registrados tres valores: **13.805** (Notebook 04), **13.965** (primera ejecución reportada de Notebook 07) y **13.972** (snapshot de salidas incluido en el ZIP actual). El snapshot actual permite verificar 13.972, pero el archivo fuente `backtesting_modelos_base.csv` y el CSV detallado original de Notebook 04 no están incluidos en el paquete recibido. Por eso no es posible reconstruir, fila a fila, las dos corridas anteriores ni atribuir la diferencia a una causa única.

La auditoría no debe ocultar esta diferencia ni declarar que se ha reconciliado. La cifra 13.972 se considera **resultado del snapshot auditado**, no una cifra históricamente reconciliada.

In [7]:
# Referencias históricas reportadas en el historial del proyecto.
historical = pd.DataFrame([
    {"origen":"Notebook 04 — resultado reportado", "pedido_Q90":13805, "evidencia_disponible":"Resumen reportado; CSV de detalle no disponible", "estado":"NO REPRODUCIBLE"},
    {"origen":"Notebook 07 — primera ejecución reportada", "pedido_Q90":13965, "evidencia_disponible":"Valor reportado; snapshot de esa ejecución no disponible", "estado":"NO REPRODUCIBLE"},
    {"origen":"ZIP Notebook 07 — snapshot incluido", "pedido_Q90":int(recommendations['pedido_sugerido'].sum()), "evidencia_disponible":"recomendacion_abastecimiento_Q90.csv y comparación Q85/Q90/Q95", "estado":"VERIFICABLE EN SNAPSHOT"}
])
display(historical)
add_check("Reconciliación histórica de corridas", historical['pedido_Q90'].nunique()==1, "Los totales históricos difieren; se requiere recuperar insumos/salidas originales", "NO RECONCILIADO")

# Revisión explícita de insumos que permiten reproducir el pipeline desde cero.
required_inputs = ['ventas_historicas.csv','inventario_actual.csv','catalogo_productos.csv','maestro_tiendas.csv','backtesting_modelos_base.csv']
input_status = pd.DataFrame([{"archivo":f,"disponible":(DATA_DIR/f).exists()} for f in required_inputs])
display(input_status)
add_check("Insumos completos para reejecutar pipeline", input_status['disponible'].all(), f"faltantes={input_status.loc[~input_status.disponible,'archivo'].tolist()}", "NO REPRODUCIBLE")

,origen,pedido_Q90,evidencia_disponible,estado
0,Notebook 04 — resultado reportado,13805,Resumen reportado; CSV de detalle no disponible,NO REPRODUCIBLE
1,Notebook 07 — primera ejecución reportada,13965,Valor reportado; snapshot de esa ejecución no ...,NO REPRODUCIBLE
2,ZIP Notebook 07 — snapshot incluido,13972,recomendacion_abastecimiento_Q90.csv y compara...,VERIFICABLE EN SNAPSHOT


,archivo,disponible
0,ventas_historicas.csv,True
1,inventario_actual.csv,True
2,catalogo_productos.csv,True
3,maestro_tiendas.csv,True
4,backtesting_modelos_base.csv,False


## 7. Resultado de auditoría y exportación

Los controles de integridad del snapshot se separan de la reconciliación histórica. Un snapshot puede ser internamente consistente y aun así no ser reproducible contra una corrida anterior si faltan los insumos originales.

In [8]:
audit = pd.DataFrame(checks)
print(audit['estado'].value_counts(dropna=False))
critical_failures = audit[audit['estado'].eq('ERROR')]
not_reconciled = audit[audit['estado'].isin(['NO RECONCILIADO','NO REPRODUCIBLE','NO EVALUADO'])]

# Métricas de referencia del snapshot actual.
q90 = recommendations[recommendations['politica']=='Q90']
metrics = {
    'combinaciones_sku_tienda': int(len(weekly)),
    'combinaciones_con_pedido_Q90': int((q90['pedido_sugerido']>0).sum()),
    'pedido_total_Q90_snapshot': int(q90['pedido_sugerido'].sum()),
    'inventario_objetivo_total_Q90': int(q90['inventario_objetivo'].sum()),
    'forecast_diario_filas': int(len(daily)),
    'residuos_semanales': int(len(errors)),
    'max_diferencia_forecast_diario_semanal': float(max_forecast_diff)
}
print(metrics)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUTPUT_DIR/'auditoria_controles.csv', index=False)
historical.to_csv(OUTPUT_DIR/'reconciliacion_Q90_historica.csv', index=False)

report = [
    '# Informe de auditoría — Tostao Supply Chain', '',
    '## Resultado',
    f"- Controles con error crítico: **{len(critical_failures)}**",
    f"- Controles pendientes/no reproducibles: **{len(not_reconciled)}**",
    f"- Pedido Q90 del snapshot: **{metrics['pedido_total_Q90_snapshot']:,} unidades**",
    f"- Combinaciones SKU–tienda: **{metrics['combinaciones_sku_tienda']}**",
    f"- Combinaciones con pedido: **{metrics['combinaciones_con_pedido_Q90']}**", '',
    '## Interpretación',
    'El snapshot debe considerarse internamente consistente solo en los controles que aparecen como OK. La diferencia entre 13.805, 13.965 y el total del snapshot no queda reconciliada con los archivos actualmente disponibles.',
    'El insumo `backtesting_modelos_base.csv` no está incluido en el paquete de proyecto recibido; por tanto, no se puede ejecutar una reproducción completa e independiente del pipeline desde sus datos de entrada.', '',
    '## Métricas del snapshot',
]
report += [f'- {k}: {v}' for k,v in metrics.items()]
report += ['', '## Controles', audit.to_markdown(index=False), '', '## Recomendación',
           'Antes de presentar los resultados como una cifra definitiva o conectarlos a Streamlit, recuperar el `backtesting_modelos_base.csv` exacto usado en la corrida de referencia, conservar los CSV de salida de Notebook 04 y volver a ejecutar Notebook 07 con una única versión de código y semilla. Guardar un identificador de corrida y los hashes de los insumos.']
(OUTPUT_DIR/'INFORME_AUDITORIA_08.md').write_text('\n'.join(report), encoding='utf-8')
print('Exportados: auditoria_controles.csv, reconciliacion_Q90_historica.csv, INFORME_AUDITORIA_08.md')

if len(critical_failures):
    raise AssertionError(f"Hay controles críticos fallidos: {critical_failures['control'].tolist()}")

estado
OK                 20
NO RECONCILIADO     1
NO REPRODUCIBLE     1
Name: count, dtype: int64
{'combinaciones_sku_tienda': 160, 'combinaciones_con_pedido_Q90': 156, 'pedido_total_Q90_snapshot': 13972, 'inventario_objetivo_total_Q90': 17286, 'forecast_diario_filas': 1120, 'residuos_semanales': 480, 'max_diferencia_forecast_diario_semanal': 5.684341886080802e-14}
Exportados: auditoria_controles.csv, reconciliacion_Q90_historica.csv, INFORME_AUDITORIA_08.md
